# A2-02: Image Segmentation with U-Net

Having covered object detection (bounding boxes), we now move to **image segmentation** — assigning a class label to *every pixel* in the image.

We will build **U-Net** from scratch and train it on the Oxford-IIIT Pet dataset.


## Types of Image Segmentation

Object detection gives us bounding boxes — but what if we need to know exactly *which pixels* belong to each object? That's where segmentation comes in.

There are two fundamentally different segmentation tasks:

| | Semantic Segmentation | Instance Segmentation |
|---|---|---|
| **Question** | What class is each pixel? | Which *individual object* does each pixel belong to? |
| **Output** | Class label per pixel | Object ID + class per pixel |
| **Same-class objects** | Merged together | Separated |
| **Typical model** | U-Net, FCN, DeepLab | Mask R-CNN |

<img src="img/segmentation_types.png" title="Semantic vs Instance Segmentation" style="width: 900px;" />


### Visual comparison

```
Original image: 🐱 🐱 🐶

Semantic segmentation:
  [cat][cat][cat][cat][dog][dog]   ← all cats same color, dog different
   ████████████████  ████████
   cat (label=1)     dog (label=2)

Instance segmentation:
  [cat#1][cat#1][cat#2][cat#2][dog#1]
   ██████  ██████  ████████
   cat id=1  cat id=2  dog id=1
```

**Key insight**: Semantic segmentation cannot distinguish between two cats sitting side by side. Instance segmentation can.

## Semantic Segmentation: FCN and U-Net

### Fully Convolutional Network (FCN, 2015)

The first major deep learning approach to semantic segmentation. The key idea: replace the final FC layers of a classification CNN with **1×1 convolutions**, then **upsample** back to input resolution.

```
Classification CNN:          FCN:
Conv → Pool → ... → FC → softmax    Conv → Pool → ... → 1×1 Conv → Upsample → pixel labels
```

Problem: upsampling from a very small feature map loses spatial detail.

### U-Net (2015)

U-Net solved the spatial detail problem with **skip connections** — direct paths from encoder to decoder:

```
Encoder (contracting path):          Decoder (expanding path):
Input (H×W×3)                        Output (H×W×C)
  → DoubleConv → 64ch    ──────────→  DoubleConv → 64ch → Conv 1×1
  → MaxPool                                              ↑ concat
  → DoubleConv → 128ch   ─────────→   DoubleConv → 128ch
  → MaxPool                                              ↑ concat
  → DoubleConv → 256ch   ────────→    DoubleConv → 256ch
  → MaxPool                                              ↑ concat
  → DoubleConv → 512ch   ───────→     DoubleConv → 512ch
  → MaxPool                                              ↑ concat
  → DoubleConv → 1024ch  ←─── Bottleneck ───────────────┘
```

**Skip connections** copy high-resolution feature maps from the encoder directly to the corresponding decoder stage. This allows the decoder to recover fine spatial details (edges, boundaries) that would otherwise be lost through pooling.

### Loss and Metrics

**Per-pixel Cross-Entropy Loss**:
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{ic} \log(\hat{p}_{ic})$$

**mean IoU (mIoU)** — standard metric for segmentation:
$$\text{mIoU} = \frac{1}{C} \sum_{c=1}^{C} \frac{TP_c}{TP_c + FP_c + FN_c}$$

where $TP_c$ = pixels correctly predicted as class $c$, $FP_c$ = pixels wrongly predicted as $c$, $FN_c$ = pixels of class $c$ missed.

## Instance Segmentation: Mask R-CNN (2017)

**Paper:** He et al., *Mask R-CNN*, ICCV 2017

Mask R-CNN extends Faster R-CNN by adding a third output head: a **pixel-level mask** for each detected instance.

### Architecture

```
Input Image
    ↓
CNN Backbone (ResNet + FPN)
    ↓
RPN → Region Proposals
    ↓
ROI Align  ← key improvement over ROI Pooling
    ├── Classification head  → class label
    ├── Bounding box head    → refined (x, y, w, h)
    └── Mask head            → 28×28 binary mask (per class)
```

### ROI Align vs ROI Pooling

ROI Pooling **quantizes** the proposal coordinates to integer pixel positions before pooling — this introduces misalignment that degrades mask quality.

ROI Align uses **bilinear interpolation** to sample features at exact (non-integer) positions, preserving spatial precision:

```
ROI Pooling:    proposal (10.7, 20.3) → snapped to (10, 20) → pool
ROI Align:      proposal (10.7, 20.3) → bilinear interp at (10.7, 20.3) → pool
```

### Mask Head

A small FCN applied independently to each ROI: 4 conv layers → predict a $K \times 28 \times 28$ mask (one binary mask per class $K$). During inference, only the mask for the predicted class is used.

### Loss

$$\mathcal{L} = \mathcal{L}_{cls} + \mathcal{L}_{box} + \mathcal{L}_{mask}$$

$\mathcal{L}_{mask}$ is **binary cross-entropy** applied only to the ground-truth class mask — so classes don't compete with each other.

### When to use which?

| Scenario | Best choice |
|---|---|
| Classify every pixel (road, sky, building) | Semantic — U-Net / DeepLab |
| Count/separate individual objects | Instance — Mask R-CNN |
| Both: per-pixel labels AND separate instances | Panoptic segmentation |

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Dataset: Oxford-IIIT Pet Dataset

**Oxford-IIIT Pet** (Parkhi et al., 2012) — 37 breeds of cats and dogs with pixel-level masks:
- **Class 1**: Pet (foreground)
- **Class 2**: Background
- **Class 3**: Border/uncertain region

**Reference:** Parkhi et al. (2012). *Cats and Dogs*. CVPR.
Dataset: https://www.robots.ox.ac.uk/~vgg/data/pets/

---

In [ ]:
from torchvision.datasets import OxfordIIITPet

os.makedirs('./data', exist_ok=True)
IMG_SIZE = 128

train_raw = OxfordIIITPet('./data', split='trainval', target_types='segmentation', download=True)
test_raw  = OxfordIIITPet('./data', split='test',     target_types='segmentation', download=True)
print(f'Train: {len(train_raw)} | Test: {len(test_raw)}')

class PetSegDataset(Dataset):
    def __init__(self, base, size=128):
        self.ds = base
        self.img_tf  = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        self.mask_tf = transforms.Compose([
            transforms.Resize((size, size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor(),
        ])
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        img, mask = self.ds[idx]
        img  = self.img_tf(img)
        mask = (self.mask_tf(mask).squeeze(0).long() - 1).clamp(0, 2)
        return img, mask

train_data   = PetSegDataset(train_raw, IMG_SIZE)
test_data    = PetSegDataset(test_raw,  IMG_SIZE)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=16, shuffle=False, num_workers=2)

# Visualize samples
mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
CLASS_COLORS = np.array([[255,100,100],[100,100,255],[255,255,100]], dtype=np.uint8)
CLASS_NAMES  = ['Pet', 'Background', 'Border']

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(3):
    img, mask = train_data[i*100]
    img_d  = torch.clamp(img * std + mean, 0, 1).permute(1,2,0).numpy()
    mask_d = CLASS_COLORS[mask.numpy()]
    axes[i][0].imshow(img_d);  axes[i][0].set_title('Image');   axes[i][0].axis('off')
    axes[i][1].imshow(mask_d); axes[i][1].set_title('Mask');    axes[i][1].axis('off')
    axes[i][2].imshow(img_d);  axes[i][2].imshow(mask_d, alpha=0.5)
    axes[i][2].set_title('Overlay'); axes[i][2].axis('off')
patches = [plt.Rectangle((0,0),1,1,color=CLASS_COLORS[i]/255) for i in range(3)]
fig.legend(patches, CLASS_NAMES, loc='lower center', ncol=3)
plt.suptitle('Oxford Pet — Image + Segmentation Mask', fontsize=13)
plt.tight_layout(); plt.show()

## Building U-Net from Scratch

The key building block is **DoubleConv**: Conv → BN → ReLU → Conv → BN → ReLU

---

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=3, features=[64,128,256,512]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        ch = features[-1] * 2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(ch, f, kernel_size=2, stride=2))
            self.decoders.append(DoubleConv(f*2, f))
            ch = f
        self.output = nn.Conv2d(features[0], n_classes, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)  # skip connection!
            x = dec(x)
        return self.output(x)


model = UNet().to(device)
dummy = torch.randn(2, 3, 128, 128).to(device)
out   = model(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}  <- same H x W as input')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## Training U-Net

Loss: **Cross-Entropy** per pixel

Metric: **mIoU** (mean Intersection over Union) — standard segmentation benchmark

$$\text{IoU} = \frac{\text{Predicted} \cap \text{Ground Truth}}{\text{Predicted} \cup \text{Ground Truth}}$$

---

In [ ]:
from tqdm import tqdm

def compute_iou(pred, target, n_classes=3):
    pred = pred.argmax(dim=1)
    ious = []
    for cls in range(n_classes):
        inter = ((pred==cls) & (target==cls)).sum().float()
        union = ((pred==cls) | (target==cls)).sum().float()
        if union > 0: ious.append((inter/union).item())
    return np.mean(ious) if ious else 0.0


EPOCHS    = 15
model     = UNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

train_losses, val_ious = [], []
for epoch in range(EPOCHS):
    model.train()
    ep_loss = []
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        imgs, masks = imgs.to(device), masks.to(device)
        loss = criterion(model(imgs), masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_loss.append(loss.item())
    model.eval()
    ep_iou = []
    with torch.no_grad():
        for imgs, masks in test_loader:
            ep_iou.append(compute_iou(model(imgs.to(device)), masks.to(device)))
    scheduler.step()
    train_losses.append(np.mean(ep_loss))
    val_ious.append(np.mean(ep_iou))
    print(f'Epoch {epoch+1:02d} | Loss: {train_losses[-1]:.4f} | mIoU: {val_ious[-1]:.4f}')

torch.save(model.state_dict(), 'unet_pet.pt')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, marker='o', color='steelblue')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)
axes[1].plot(val_ious, marker='s', color='darkorange')
axes[1].set_title('Validation mIoU'); axes[1].set_xlabel('Epoch'); axes[1].grid(True)
plt.tight_layout(); plt.show()
print(f'Best mIoU: {max(val_ious):.4f}')

## Visualize Predictions

---

In [ ]:
model.load_state_dict(torch.load('unet_pet.pt', map_location=device))
model.eval()

fig, axes = plt.subplots(5, 4, figsize=(14, 18))
for ax, t in zip(axes[0], ['Input','Ground Truth','Prediction','Overlay']):
    ax.set_title(t, fontsize=11, fontweight='bold')

for row in range(5):
    img, mask = test_data[row*50]
    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
    img_d  = torch.clamp(img * std + mean, 0, 1).permute(1,2,0).numpy()
    axes[row][0].imshow(img_d)
    axes[row][1].imshow(CLASS_COLORS[mask.numpy()])
    axes[row][2].imshow(CLASS_COLORS[pred_mask])
    axes[row][3].imshow(img_d); axes[row][3].imshow(CLASS_COLORS[pred_mask], alpha=0.5)
    for ax in axes[row]: ax.axis('off')

patches = [plt.Rectangle((0,0),1,1,color=CLASS_COLORS[i]/255) for i in range(3)]
fig.legend(patches, CLASS_NAMES, loc='lower center', ncol=3)
plt.suptitle('U-Net Results — Oxford Pet Dataset', fontsize=13)
plt.tight_layout(); plt.show()

# Exercises

## Exercise 1

1. **Skip Connections — Do They Matter?**

   a) Train a U-Net **without skip connections** — modify `forward()` so the decoder never concatenates encoder feature maps. Also update decoder input channels from `f*2` to `f`.

   b) Compare mIoU:

   | Model | Val mIoU | Time/epoch |
   |---|---|---|
   | U-Net with skip connections | ? | ? |
   | U-Net without skip connections | ? | ? |

   c) Why do skip connections help segmentation more than they would help classification?

2. **Pretrained ResNet Encoder**

   Modern U-Nets use a pretrained CNN as encoder instead of training from scratch.

   a) Replace the U-Net encoder with pretrained **ResNet-18**:
   ```python
   import torchvision.models as models
   resnet = models.resnet18(weights='IMAGENET1K_V1')
   # Use resnet.layer1, layer2, layer3, layer4 as encoder stages
   ```

   b) Compare:

   | Model | # Params | Val mIoU | Time/epoch |
   |---|---|---|---|
   | U-Net from scratch | ? | ? | ? |
   | U-Net + ResNet-18 encoder | ? | ? | ? |

   c) Why does ImageNet pretraining help even though ImageNet is a classification dataset?

3. **YOLO + U-Net Pipeline**

   Real systems often combine detection and segmentation:

   a) Using the YOLO model from A2-01, detect objects in a sample image and crop each bounding box region.

   b) Resize each crop to 128×128 and feed into your trained U-Net.

   c) Visualize the result: YOLO bounding box + U-Net segmentation mask overlaid inside each box.

4. **(Challenge) Binary Segmentation**

   Simplify to **2 classes** (pet vs. background) and compare with the 3-class model.

   a) Modify the dataset loader to merge classes:
   ```python
   mask[mask > 0] = 1   # foreground=1, background=0
   ```

   b) Use Binary Cross-Entropy loss. Compute **Dice Score**:
   $$\text{Dice} = \frac{2 |P \cap G|}{|P| + |G|}$$

   c) Compare 3-class vs 2-class:

   | Setting | Val mIoU | Dice Score |
   |---|---|---|
   | 3-class (pet / cat / dog) | ? | ? |
   | 2-class (pet vs background) | ? | ? |

---

## Submission

Submit your work to GitHub. Your repository should contain:

### 1. Training Script (`train.py`)

```bash
# Train U-Net from scratch
python3 train.py --model unet          --dataset oxford_pet --epochs 20 --train

# Train with pretrained ResNet encoder
python3 train.py --model unet_resnet18 --dataset oxford_pet --epochs 20 --train

# Train without skip connections (ablation)
python3 train.py --model unet_no_skip  --dataset oxford_pet --epochs 20 --train

# Evaluate saved model
python3 train.py --model unet --weights unet_pet.pt --dataset oxford_pet --evaluate
```

### 2. `README.md`

Your `README.md` must include:

**Commands used** (exact commands you ran)

**Results table:**

| Model | # Params | Val mIoU | Time/epoch | Notes |
|---|---|---|---|---|
| U-Net (from scratch) | ? | ? | ? | baseline |
| U-Net (no skip connections) | ? | ? | ? | ablation |
| U-Net + ResNet-18 encoder | ? | ? | ? | pretrained encoder |

**Discussion** (3–5 sentences): Which model performed best? When would you choose U-Net over Mask R-CNN?